In [1]:
import pandas as pd
import utils
import model_generators as mg
import datetime

from dateutil.relativedelta import relativedelta

from dataclasses import dataclass

from enum import Enum
from __future__ import annotations

c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [2]:
@dataclass()
class Config():
    """Classe para a definicao das variaveis de ambiente"""
    horizonte_previsao: int
    tamanho_teste: int
    n_trials_skus: int
    metrica_erro: Status
    tolerancia_fipe: int
    tolerancia_exog: float
    skus: list[int]
    data_ref: datetime.datetime
    dt_inicio_teste: datetime.datetime
    dt_limite_inferior_exog: datetime.datetime
    exog_list: list[str]

    def __init__(self, horizonte_previsao, tamanho_teste, n_trials_skus, n_trials_exog, metrica_erro, tolerancia_fipe, tolerancia_exog, skus, data_ref, dt_limite_inferior_exog, exog_list):
        self.horizonte_previsao = horizonte_previsao
        self.tamanho_teste = tamanho_teste
        self.n_trials_skus = n_trials_skus
        self.n_trials_exog = n_trials_exog
        self.metrica_erro = metrica_erro
        self.tolerancia_fipe = tolerancia_fipe
        self.tolerancia_exog = tolerancia_exog
        self.skus = skus
        self.data_ref = data_ref
        self.dt_limite_inferior_exog = dt_limite_inferior_exog
        self.exog_list = exog_list
        self.dt_inicio_teste = data_ref + relativedelta(months=-tamanho_teste)

class Status(Enum):
    MAE = 'mae'
    MAPE = 'mape'

In [3]:
config = Config(
    horizonte_previsao = 3,
    tamanho_teste = 6,
    n_trials_skus = 1,
    n_trials_exog = 1,
    metrica_erro = Status.MAE,
    tolerancia_fipe = 200,
    tolerancia_exog = 0.01,
    skus = [100, 1832, 2134, 5112, 7023],
    data_ref = datetime.datetime(2026, 5, 1),
    dt_limite_inferior_exog = datetime.datetime(int(pd.to_datetime('today').year - 10), 1, 1),
    exog_list = ['valor', 'exchange_rate']
)

In [4]:
# Base fipe historica
fipe_path = './data/dados_fipe_tratados.csv'
# Base da taxa de cambio
exchange_path = './data/DEXBZUS_tratados.csv'
# Base IPCA
ipca_path = './data/bcdata.sgs.433_tratados.csv' 

df_fipe = pd.read_csv(fipe_path).drop(columns=['Unnamed: 0', 'year_of_reference', 'month_of_reference'])

df_exchange = pd.read_csv(exchange_path)

df_ipca = pd.read_csv(ipca_path).drop(columns=['data', 'ipca'])

In [5]:
df_ipca['date'] = pd.to_datetime(df_ipca['date'])
df_exchange['date'] = pd.to_datetime(df_exchange['date'])
df_fipe['reference_date'] = pd.to_datetime(df_fipe['reference_date'], format='ISO8601')

df_ipca.index = df_ipca['date']
df_exchange.index = df_exchange['date']

In [6]:
# Criacao dos skus
# Refazer essa logica para ele ir incrementando
df_fipe['sku'] = df_fipe.groupby(['brand_name', 'model_name', 'fuel_name', 'year']).ngroup()

In [7]:
display(df_fipe.head())

,reference_date,brand_name,model_name,year,fuel_name,brl_price,sku
0,2021-01-01,Fiat,147 C/ CL,1987,Gasolina,2723.0,2
1,2021-01-01,Fiat,147 C/ CL,1986,Gasolina,2484.0,1
2,2021-01-01,Fiat,147 C/ CL,1985,Gasolina,2324.0,0
3,2021-01-01,Fiat,147 Furgão (todos),1987,Gasolina,2199.0,5
4,2021-01-01,Fiat,147 Furgão (todos),1986,Gasolina,2094.0,4


In [8]:
df_previsao_skus = utils.coletar_base_skus(config.data_ref, config.skus, df_fipe)

## Simulação da solução

A simulação consiste em estimar o lucro que seria obtido pelo cliente se usasse nossa solução para tomada de decisões de compra e venda para um conjunto de carros, para esse exemplo serão considerados 5 carros por um período de 6 meses, onde o modelo prevê um horizonte de um mês e o cliente toma a decisão para o esse mês com base na previsão, depois o modelo é retreinado e é simulado o próximo mês até bater os 6 meses.

### Política:

- Comprar o carro pelo preço atual se a previsão for de subida e for maior em pelo menos 0,1% do valor atual.
- Vender:
    - Pelo preço atual se a previsão for de queda e o valor for menor em pelo menos 0,1% do valor comprado.
    - E o preço atual for maior que o preço de compra.

### Valores iniciais
* Saldo: R$ 500.000
* Estoque de carros: 0 carros
* Horizonte de previsão: 1 mês

In [21]:
saldo_inicial = 500000
saldo = saldo_inicial

# Carro = (estoque, preco_compra)
carros = {k: [] for k in config.skus}
estoque_inicial = {k: len(v) for k, v in carros.items()}

start_sim = config.data_ref + relativedelta(months=-6) # 2025-11-01
end_sim = start_sim + relativedelta(months=6) # 2026-04-01
curr_sim = start_sim

mensagens = ''
historico_transacoes = []

while curr_sim < end_sim:
    data_formatada = curr_sim.strftime('%Y-%m-%d')
    start = curr_sim + relativedelta(months=-3)

    # Treinar modelo para exogenas
    ## Separar dados de treino e teste
    df_exog_train, df_exog_test = utils.separar_treino_teste_exogena(df_ipca, df_exchange, start, config.dt_limite_inferior_exog)

    ## Treinar modelo do cambio
    ### Treinar modelo Sarimax
    model_exchange_sarimax, info_exchange_sarimax, best_value_exchange_sarimax = mg.generate_sarimax_model(
        df_exog_train['exchange_rate'],
        df_exog_test['exchange_rate'],
        None,
        None,
        config.n_trials_exog,
        config.metrica_erro,
        config.tolerancia_exog
    )

    model_exchange_sarimax_fit = model_exchange_sarimax.fit(disp=False)

    forecasts_exchange_sarimax = model_exchange_sarimax_fit.forecast(steps=len(df_exog_test['exchange_rate']))

    ### Treinar modelo Prophet
    df_prophet_train_exchange, df_prophet_test_exchange = utils.criar_dataset_prophet(df_exog_train, df_exog_test, 'date', 'exchange_rate', [])

    model_exchange_prophet, info_exchange_prophet, best_value_exchange_prophet = mg.generate_prophet_model(
        df_prophet_train_exchange,
        df_prophet_test_exchange,
        [],
        config.n_trials_exog,
        config.metrica_erro,
        config.tolerancia_exog
    )

    model_exchange_prophet.fit(
        df_prophet_train_exchange,
    )

    forecast_exchange = model_exchange_prophet.predict(
        df_prophet_test_exchange[['ds']]
    )

    ## Treinar modelo do IPCA
    ### Treinar modelo Sarimax
    model_ipca_sarimax, info_ipca_sarimax, best_value_ipca_sarimax = mg.generate_sarimax_model(
        df_exog_train['valor'],
        df_exog_test['valor'],
        None,
        None,
        config.n_trials_exog,
        config.metrica_erro,
        config.tolerancia_exog
    )

    model_ipca_sarimax_fit = model_ipca_sarimax.fit(disp=False)

    forecasts_ipca_sarimax = model_ipca_sarimax_fit.forecast(steps=len(df_exog_test['valor']))

    ### Treinar modelo Prophet
    df_prophet_train_ipca, df_prophet_test_ipca = utils.criar_dataset_prophet(df_exog_train, df_exog_test, 'date', 'valor', [])

    model_ipca_prophet, info_ipca_prophet, best_value_ipca_prophet = mg.generate_prophet_model(
        df_prophet_train_ipca,
        df_prophet_test_ipca,
        [],
        config.n_trials_exog,
        config.metrica_erro,
        config.tolerancia_exog
    )

    model_ipca_prophet.fit(
        df_prophet_train_ipca,
    )

    forecast_ipca = model_ipca_prophet.predict(
        df_prophet_test_ipca[['ds']]
    )

    # Selecionar o melhor modelo para cada exogena
    dict_kwargs = {
        'best_value_ipca_prophet': best_value_ipca_prophet,
        'best_value_ipca_sarimax': best_value_ipca_sarimax,
        'best_value_exchange_prophet': best_value_exchange_prophet,
        'best_value_exchange_sarimax': best_value_exchange_sarimax,
        'info_ipca_prophet': info_ipca_prophet,
        'info_ipca_sarimax': info_ipca_sarimax,
        'info_exchange_prophet': info_exchange_prophet,
        'info_exchange_sarimax': info_exchange_sarimax,
        'df_prophet_train_ipca': df_prophet_train_ipca,
        'df_prophet_test_ipca': df_prophet_test_ipca,
        'df_prophet_train_exchange': df_prophet_train_exchange,
        'df_prophet_test_exchange': df_prophet_test_exchange,
        'df_exog_train': df_exog_train,
        'df_exog_test': df_exog_test
    }

    df_exog_previsao, modelo_escolhido_ipca, modelo_escolhido_exchange = utils.selecionar_modelo_exog(
        horizonte_previsao=config.horizonte_previsao,
        **dict_kwargs
    )

    previsoes_por_sku = {}
    modelo_vencedor_por_sku = {}
    for sku in config.skus:
        df_sku_atual = df_previsao_skus.query('sku == @sku')

        if df_sku_atual.empty:
            print(f"Pulando SKU: {sku} por poucos dados...")
            continue

        df_train_sku, df_test_sku = utils.separar_treino_teste_sku(df_sku_atual, curr_sim, start)

        df_exog_train_sku = df_exog_train[df_exog_train.index.isin(df_train_sku['reference_date'])].copy()
        df_exog_test_sku = df_exog_test.copy()

        df_exog_train_sku = df_exog_train_sku[config.exog_list]
        df_exog_test_sku = df_exog_test_sku[config.exog_list]

        df_train_prophet_sku, df_test_prophet_sku = utils.criar_dataset_prophet(
            pd.concat([df_train_sku, df_exog_train_sku], axis=1),
            pd.concat([df_test_sku, df_exog_test], axis=1),
            'reference_date',
            'brl_price',
            config.exog_list
        )

        dict_modelos_sku = mg.criar_modelos_fipe(
            df_train_sku,
            df_test_sku,
            df_train_prophet_sku,
            df_test_prophet_sku,
            df_exog_train_sku,
            df_exog_test_sku,
            config.n_trials_skus,
            config.metrica_erro,
            config.tolerancia_fipe,
            'brl_price'
        )

        forecast_sku, modelo_escolhido_sku = utils.selecionar_modelo_fipe(
            df_train_sku,
            df_test_sku,
            df_train_prophet_sku,
            df_test_prophet_sku,
            df_exog_train_sku,
            df_exog_test_sku,
            df_exog_previsao,
            config.horizonte_previsao,
            'brl_price',
            **dict_modelos_sku
        )

        previsoes_por_sku[sku] = forecast_sku
        modelo_vencedor_por_sku[sku] = modelo_escolhido_sku

    # Política do cliente
    for sku, carro in carros.items():
        preco_atual = df_previsao_skus.query('sku == @sku and reference_date == @curr_sim')['brl_price'].iloc[0]
        preco_previsto = previsoes_por_sku[sku].iloc[0]

        # Comprar
        if preco_previsto >= preco_atual*1.001 and saldo >= preco_atual:
            mensagem = f"[{data_formatada}] Comprou o sku {sku} por R$ {preco_atual:,.2f}\n"
            print(mensagem)
            mensagens += mensagem
            saldo -= preco_atual
            carros[sku].append((1, preco_atual))
            
            historico_transacoes.append({
                'data': data_formatada,
                'tipo': 'COMPRA',
                'sku': sku,
                'preco': preco_atual,
                'custo_compra': preco_atual,
                'lucro': 0.0
            })
        
        # Vender
        carros_mantidos = []
        for estoque, preco_compra in carro:
            if (preco_previsto <= preco_compra * 0.999) and (preco_atual > preco_compra):
                lucro_operacao = preco_atual - preco_compra
                mensagem = f"[{data_formatada}] Vendeu o sku {sku} por R$ {preco_atual:,.2f} (Custo: R$ {preco_compra:,.2f} | Lucro: R$ {lucro_operacao:,.2f})\n"
                print(mensagem)
                mensagens += mensagem
                saldo += preco_atual
                
                historico_transacoes.append({
                    'data': data_formatada,
                    'tipo': 'VENDA',
                    'sku': sku,
                    'preco': preco_atual,
                    'custo_compra': preco_compra,
                    'lucro': lucro_operacao
                })
            else:
                carros_mantidos.append((estoque, preco_compra))
        
        carros[sku] = carros_mantidos
        
    curr_sim += relativedelta(months=1)

[I 2026-06-29 18:16:58,041] A new study created in memory with name: no-name-9520753c-24c7-4be7-a592-37214f53b8ae
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
[I 2026-06-29 18:16:58,364] Trial 0 finished with value: 0.16283771320030083 and parameters: {'p': 2, 'd': 2, 'q': 0, 'trend': 'c', 'seasonal': True, 'P': 2, 'D': 0, 'Q': 1}. Best is trial 0 with value: 0.16283771320030083.
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used

[2026-01-01] Comprou o sku 5112 por R$ 89,184.00



[I 2026-06-29 18:17:26,605] A new study created in memory with name: no-name-4e96d013-dd89-470f-af40-0b17cef3199f
18:17:26 - cmdstanpy - INFO - Chain [1] start processing
18:17:26 - cmdstanpy - INFO - Chain [1] done processing
[I 2026-06-29 18:17:26,772] Trial 0 finished with value: 0.665262434774447 and parameters: {'changepoint_prior_scale': 0.12106126234767979, 'seasonality_prior_scale': 0.08867868442555407, 'seasonality_mode': 'additive', 'changepoint_range': 0.8650103504383285, 'n_changepoints': 21}. Best is trial 0 with value: 0.665262434774447.
18:17:26 - cmdstanpy - INFO - Chain [1] start processing
18:17:26 - cmdstanpy - INFO - Chain [1] done processing
[I 2026-06-29 18:17:26,937] A new study created in memory with name: no-name-73db071b-f084-48f0-9959-f260555a7565
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_da

[2026-03-01] Comprou o sku 100 por R$ 67,293.00

[2026-03-01] Comprou o sku 1832 por R$ 65,275.00



[I 2026-06-29 18:17:48,963] Trial 0 finished with value: 0.1429474598566262 and parameters: {'p': 5, 'd': 2, 'q': 5, 'trend': 'ct', 'seasonal': True, 'P': 1, 'D': 0, 'Q': 2}. Best is trial 0 with value: 0.1429474598566262.
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
[I 2026-06-29 18:17:49,329] A new study created in memory with name: no-name-33e18a95-b2e5-473b-951b-d7fb50212657
18:17:49 - cmdstanpy - INFO - Chain [1] start processing
18:17:49 - cmdstanpy - INFO - Chain [1] done processing
[I 2026-06-29 18:17:49,514] Trial 0 finished with value: 0.8487253121332063 and parame

[2026-04-01] Comprou o sku 5112 por R$ 88,004.00



c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


In [24]:
df_transacoes = pd.DataFrame(historico_transacoes)

estoque_final = {k: len(v) for k, v in carros.items()}
capital_preso_estoque = sum(sum(preco_compra for _, preco_compra in lista_carros) for lista_carros in carros.values())

lucro_realizado = df_transacoes[df_transacoes['tipo'] == 'VENDA']['lucro'].sum() if not df_transacoes.empty else 0.0

patrimonio_inicial = saldo_inicial
patrimonio_final = saldo + capital_preso_estoque
lucro_total_patrimonial = patrimonio_final - patrimonio_inicial
roi_total = (lucro_total_patrimonial / patrimonio_inicial) * 100

total_compras = len(df_transacoes[df_transacoes['tipo'] == 'COMPRA']) if not df_transacoes.empty else 0
total_vendas = len(df_transacoes[df_transacoes['tipo'] == 'VENDA']) if not df_transacoes.empty else 0

relatorio = f"""
================================================================================
                    RELATÓRIO DE SIMULAÇÃO DE TRADING
================================================================================

Pereíodo da Simulação: {start_sim.strftime('%Y-%m-%d')} até {(end_sim - relativedelta(months=1)).strftime('%Y-%m-%d')}
--------------------------------------------------------------------------------

[1] RESUMO FINANCEIRO E PERFORMANCE
    - Saldo Caixa Inicial:       R$ {saldo_inicial:,.2f}
    - Saldo Caixa Final:         R$ {saldo:,.2f}
    - Valor em Estoque Final:    R$ {capital_preso_estoque:,.2f} (Preço de Custo)
    
    - Patrimônio Líquido Inicial:R$ {patrimonio_inicial:,.2f}
    - Patrimônio Líquido Final:  R$ {patrimonio_final:,.2f}
    
    - Lucro Caixa Realizado:     R$ {lucro_realizado:,.2f} (Apenas Vendas)
    - Lucro Patrimonial Total:   R$ {lucro_total_patrimonial:,.2f} (Caixa + Estoque Atual)
    - Retorno s/ Investimento:   {roi_total:.2f}%

[2] VOLUMETRIA DE TRANSAÇÕES
    - Total de Operações:        {total_compras + total_vendas}
    - Carros Comprados:          {total_compras}
    - Carros Vendidos:           {total_vendas}

[3] CONTROLE DE ESTOQUE (Unidades)
"""

for sku in config.skus:
    relatorio += f"    - SKU {sku}: Inicial ({estoque_inicial[sku]}) | Final ({estoque_final[sku]})\n"

relatorio += "\n[4] REGISTRO DETALHADO DE TRANSAÇÕES (Ordem Cronológica)\n"
if not df_transacoes.empty:
    relatorio += df_transacoes[['data', 'tipo', 'sku', 'preco', 'custo_compra', 'lucro']].to_string(index=False, formatters={
        'preco': lambda x: f"R$ {x:,.2f}",
        'custo_compra': lambda x: f"R$ {x:,.2f}",
        'lucro': lambda x: f"R$ {x:,.2f}" if x > 0 else "-"
    })
else:
    relatorio += "    Nenhuma transação foi efetuada no período."

relatorio += "\n================================================================================"

print(relatorio)


                    RELATÓRIO DE SIMULAÇÃO DE TRADING

Pereíodo da Simulação: 2025-11-01 até 2026-04-01
--------------------------------------------------------------------------------

[1] RESUMO FINANCEIRO E PERFORMANCE
    - Saldo Caixa Inicial:       R$ 500,000.00
    - Saldo Caixa Final:         R$ 190,244.00
    - Valor em Estoque Final:    R$ 309,756.00 (Preço de Custo)

    - Patrimônio Líquido Inicial:R$ 500,000.00
    - Patrimônio Líquido Final:  R$ 500,000.00

    - Lucro Caixa Realizado:     R$ 0.00 (Apenas Vendas)
    - Lucro Patrimonial Total:   R$ 0.00 (Caixa + Estoque Atual)
    - Retorno s/ Investimento:   0.00%

[2] VOLUMETRIA DE TRANSAÇÕES
    - Total de Operações:        4
    - Carros Comprados:          4
    - Carros Vendidos:           0

[3] CONTROLE DE ESTOQUE (Unidades)
    - SKU 100: Inicial (0) | Final (1)
    - SKU 1832: Inicial (0) | Final (1)
    - SKU 2134: Inicial (0) | Final (0)
    - SKU 5112: Inicial (0) | Final (2)
    - SKU 7023: Inicial (0) | Fi